# DC Motor Digital Twin — Model Development

## Objective

This project develops a physics-based digital twin of a permanent-magnet DC motor.

The initial objective is to:

1. derive the electrical and mechanical motor equations;
2. formulate the model in state-space form;
3. implement the model in Python;
4. simulate the open-loop response to a voltage step;
5. compare simulation results with analytical steady-state predictions.

The project will subsequently be extended toward closed-loop control,
system identification, parameter estimation, and digital-twin development.


## 1. Electrical Model

The armature voltage equation is

$$
V_a(t) = L\frac{di(t)}{dt} + Ri(t) + K_e\omega(t)
$$

Therefore,

$$
\frac{di}{dt}
=
-\frac{R}{L}i
-\frac{K_e}{L}\omega
+\frac{1}{L}V_a
$$


## 2. Mechanical Model

The mechanical dynamics are

$$
J\frac{d\omega}{dt}
=
K_t i-b\omega-\tau_L
$$

Therefore,

$$
\frac{d\omega}{dt}
=
\frac{K_t}{J}i
-\frac{b}{J}\omega
-\frac{1}{J}\tau_L
$$


## 3. State-Space Representation

Define the state vector as

$$
x =
\begin{bmatrix}
i\\
\omega
\end{bmatrix}
$$

The model is represented as

$$
\dot{x}=Ax+Bu+E\tau_L
$$

where

$$
A =
\begin{bmatrix}
-\frac{R}{L} & -\frac{K_e}{L}\\
\frac{K_t}{J} & -\frac{b}{J}
\end{bmatrix}
$$

and

$$
B =
\begin{bmatrix}
\frac{1}{L}\\
0
\end{bmatrix}
$$

$$
E =
\begin{bmatrix}
0\\
-\frac{1}{J}
\end{bmatrix}
$$


from src.motor import DCMotor

motor = DCMotor()

A, B, E = motor.matrices()

print("A =")
print(A)

print("\nB =")
print(B)

print("\nE =")
print(E)


import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

from src.motor import DCMotor

motor = DCMotor()

voltage = 12.0
t_start = 0.0
t_end = 15.0

def dynamics(t, x):
    return motor.derivatives(
        x,
        voltage=voltage,
        load_torque=0.0,
    )

x0 = np.array([0.0, 0.0])

t_eval = np.linspace(t_start, t_end, 1000)

solution = solve_ivp(
    dynamics,
    (t_start, t_end),
    x0,
    t_eval=t_eval,
    rtol=1e-8,
    atol=1e-10,
)

current = solution.y[0]
speed = solution.y[1]


## Analytical Steady-State Validation

At steady state,

$$
\frac{di}{dt}=0
$$

and

$$
\frac{d\omega}{dt}=0
$$

For zero load torque,

$$
i_{ss}=\frac{b}{K_t}\omega_{ss}
$$

and therefore

$$
\omega_{ss}
=
\frac{V_a}
{K_e+\frac{Rb}{K_t}}
$$

For the nominal parameters and a 12 V input:

$$
\omega_{ss}=40\;rad/s
$$

The corresponding steady-state current is

$$
i_{ss}=4\;A
$$
